In [17]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm, uniform
import plotly.graph_objects as go
N = 2976

def exp( x , a, A, tau ):
    return a*N*expon.cdf(x , A , tau) 


def exp_unif(x, a, b, tau, A):
    return a * N * (expon.cdf(x, A, tau) + b * N * uniform.cdf(x, 0, 8e-6))

# def exp_gauss_cdf(N_exp , A , tau):
#     def fixed_exp( x , N , mu , sigma):
#         return exp( x , N_exp , A , tau) + gauss( x , N , mu , sigma)
#     return fixed_exp
    
# def gauss_exp_cdf(N_g , mu , sigma):
#     def fixed_gauss( x , N , A , tau):
#         return exp( x , N , A , tau) + gauss( x , N_g , mu , sigma)
#     return fixed_gauss

def exp_fit(cdf, a, A, tau, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, A, tau )
    n.fixed["A"] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def exp_unif_fit(cdf, a, b, tau, A, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, b, tau, A )
    # n.fixed['N'] = True
    n.fixed['A'] = True
    # n.limits['b'] = (0, 1)
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

In [2]:
data_1 = np.genfromtxt("Data/timestamp/21_01_2024_14_40_PMG.csv", delimiter=',')
# print(len(data_1))
data_2 = np.genfromtxt("Data/timestamp/21_01_2024_15_50_low_M_Voltage.csv", delimiter=',')
# # print(len(data_2))
data_3 = np.genfromtxt("Data/timestamp/21_01_2024_17_42.csv", delimiter=',')
# print(len(data_3))
n_bins = 80
# data_2 = data_2[(data_2 > 0.6e-6) & (data_2 < 1.8e-6)]
count, edges = np.histogram(data_1, bins=n_bins , density=False)
count_2, edges_2 = np.histogram(data_2, bins=n_bins , density=True)
count_3, edges_3 = np.histogram(data_3, bins=n_bins , density=False)

fig = go.Figure()

# fig.add_trace(go.Bar(x=edges[:-1], y=count, name='Histogram', width=np.diff(edges)))
# fig.add_trace(go.Bar(x=edges_2[:-1], y=count_2, name='Histogram 2', width=np.diff(edges_2)))
fig.add_trace(go.Bar(x=edges_3[:-1], y=count_3, name='Histogram 3', width=np.diff(edges_3)))

fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()
print("sum counts:", np.sum(count_3))

sum counts: 2976


In [18]:
k = exp_fit( exp, 1, 0, 2.2e-6, count_3 , edges_3)
display(k)


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 114.1 (χ²/ndof = 1.5)      │              Nfcn = 35               │
│ EDM = 1.74e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.026   │   0.019   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │  2.21e-6  │  0.05e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬────────────────────────────────────────┐
│     │            a            A          tau │
├─────┼────────────────────────────────────────┤
│   a │     0.000359            0 115.2477e-12 │
│   A │            0            0            0 │
│ tau │ 115.2477e-12            0     2.58e-15 │
└─────┴────────────────────────────────────────┘

In [19]:
k = exp_unif_fit( exp_unif, 1, 2e-3, 2.2e-6, 0, count_3 , edges_3)
display(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 96.6 (χ²/ndof = 1.3)       │              Nfcn = 279              │
│ EDM = 0.000152 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.20    │   0.05    │            │            │         │         │       │
│ 1 │ b    │  -40e-6   │   9e-6    │            │            │         │         │       │
│ 2 │ tau  │  2.65e-6  │  0.14e-6  │            │            │         │         │       │
│ 3 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────────────────┐
│     │           a           b         tau           A │
├─────┼─────────────────────────────────────────────────┤
│   a │     0.00271  -410.08e-9 6.054059e-9      0.0000 │
│   b │  -410.08e-9    7.61e-11  -1.085e-12           0 │
│ tau │ 6.054059e-9  -1.085e-12    1.81e-14           0 │
│   A │      0.0000           0           0           0 │
└─────┴─────────────────────────────────────────────────┘